In [4]:
from sklearn.model_selection import train_test_split ,RandomizedSearchCV
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.metrics import r2_score , log_loss, root_mean_squared_error, confusion_matrix, r2_score, mean_squared_error
from sklearn.linear_model import Ridge
from sklearn.ensemble import RandomForestRegressor
from xgboost import XGBRegressor
import pandas as pd
import numpy as np
import joblib as jbl
# from pip_trainning import pip_test
# import matplotlib.pyplot as plt


In [2]:
csv_path = r"Data\vic\REkengineered.csv"


In [3]:
data=pd.read_csv(csv_path, sep=',')


In [4]:
datas = data.select_dtypes(include=['str']).columns
datan = data.select_dtypes(include=['int64', 'float64']).columns
print(datas)
print(datan)

Index(['Location', 'Fuel_Type', 'Transmission', 'Owner_Type', 'Model',
       'Brand'],
      dtype='str')
Index(['Year', 'Kilometers_Driven', 'Mileage', 'Engine', 'Price'], dtype='str')


In [5]:


num= ['Year', 'Kilometers_Driven', 'Mileage', 'Engine']
cat = ['Location', 'Fuel_Type', 'Transmission', 'Owner_Type', 'Model',
       'Brand']

In [21]:
Grid_pipe_xg= {
    "model__n_estimators": [100, 200, 300,500,1000,2000],
    "model__max_depth": [3, 5, 7, 10],
    "model__learning_rate": [0.01, 0.05, 0.1, 0.2, 0.3],
    "model__subsample": [0.8, 1.0, 0.5],
    "model__colsample_bytree": [0.8, 1.0, 0.5],
    "model__gamma": [0, 0.1, 0.3, 0.5],
    "model__reg_alpha": [0, 0.1, 0.5],
    "model__reg_lambda": [1, 5, 10]
}

Grid_pipe_random = {
"model__n_estimators": [100,200,300],
"model__max_depth" : [None, 10,20],
"model__min_samples_split": [2,5,8],
"model__min_samples_leaf":[1,2,4],

}

Grid_pipe_ridge = {
    "model__alpha": [0.001, 0.01, 0.1, 1, 10, 100, 1000],
    "model__solver": ["auto","lsqr"]
}

cx ={
    'model__subsample': [0.8],
    'model__reg_lambda': [10],
    'model__reg_alpha': [0],
    'model__n_estimators': [1000],
    'model__max_depth': [5],
    'model__learning_rate': [0.3],
    'model__gamma': [0.5],
    'model__colsample_bytree': [1.0]
}

In [111]:
RD=pip_test(csv_path, model=RandomForestRegressor(random_state=40), grid=Grid_pipe_random, num=num, cat=cat)
print(RD[-1],RD[3])

Fitting 5 folds for each of 10 candidates, totalling 50 fits
3.0718466485325537 0.9264282466278163


In [22]:
XG=pip_test(csv_path, model=XGBRegressor(objective="reg:squarederror", random_state=42), grid=cx, num=num, cat=cat)
print(XG[1])
print(XG[-1],XG[3])
model = XG[2]

c:\Users\mohammad.karimi\Desktop\github\Car\Car_Price\.ven\Lib\site-packages\sklearn\model_selection\_search.py:324: UserWarning: The total space of parameters 1 is smaller than n_iter=10. Running 1 iterations. For exhaustive searches, use GridSearchCV.
  warnings.warn(


Fitting 5 folds for each of 1 candidates, totalling 5 fits
{'model__subsample': 0.8, 'model__reg_lambda': 10, 'model__reg_alpha': 0, 'model__n_estimators': 1000, 'model__max_depth': 5, 'model__learning_rate': 0.3, 'model__gamma': 0.5, 'model__colsample_bytree': 1.0}
2.1372250929352323 0.9324950601169718


In [23]:
data=pd.read_csv(r"Data\vic\REkengineered.csv", sep=',')
x=data.drop(columns="Price")
y=data["Price"]

X_train,X_test,y_train,y_test=train_test_split(x,y,test_size=0.2, random_state=42)
# 3️⃣ Train model with evaluation metrics
eval_set = [(X_train, y_train), (X_test, y_test)]
# model.fit(
#     X_train, y_train,
#     eval_set=eval_set,
#     eval_metric=["rmse", "mae"],  # track multiple metrics
#     early_stopping_rounds=10,
#     verbose=False
# )

# 4️⃣ Get evaluation results
results = model.evals_result()  # Get evaluation results for all metrics

# 5️⃣ Plot learning curves
plt.figure(figsize=(12, 6))

# RMSE
plt.plot(results['validation_0']['rmse'], label='Train RMSE', color='blue')
plt.plot(results['validation_1']['rmse'], label='Validation RMSE', color='orange')

# MAE
plt.plot(results['validation_0']['mae'], label='Train MAE', color='green', linestyle='--')
plt.plot(results['validation_1']['mae'], label='Validation MAE', color='red', linestyle='--')

plt.xlabel('Boosting Round')
plt.ylabel('Error')
plt.title('XGBoost Regression Learning Curves')
plt.legend()
plt.grid(True)
plt.show()


AttributeError: 'Pipeline' object has no attribute 'evals_result'

In [85]:
Ridge=pip_test(csv_path, model=Ridge(), grid=Grid_pipe_ridge, num=num, cat=cat)
print(Ridge[-1],Ridge[3])

TypeError: 'tuple' object is not callable

In [78]:
dat=pd.read_csv(csv_path, sep=',')
X = dat.drop(columns=['Price'])
y = dat['Price']
X_train, X_val, y_train, y_val = train_test_split(X, y, test_size=0.2, random_state=42)

best_params = XG[1]

final_model = XGBRegressor(
    **best_params,
    n_estimators=3000,
    objective="reg:squarederror",
    random_state=42,
    early_stopping_rounds=100
)

final_model.fit(
    X_train, y_train,
    eval_set=[(X_val, y_val)],
    verbose=False
)

ValueError: DataFrame.dtypes for data must be int, float, bool or category. When categorical type is supplied, the experimental DMatrix parameter`enable_categorical` must be set to `True`.  Invalid columns:Location: str, Fuel_Type: str, Transmission: str, Owner_Type: str, Model: str, Brand: str

In [5]:
dtf=pd.read_csv(r'Data\gtengineered.csv', sep=',')

In [24]:
dtf.head()

,Location,Year,Kilometers_Driven,Fuel_Type,Transmission,Owner_Type,Consommation,Engine,Price,Brand,Model
0,Pune,2015,41000,Diesel,Manual,First,19.67,1582,12.50,Hyundai,Creta 1.6 CRDi SX Option
1,Chennai,2011,46000,Petrol,Manual,First,18.20,1199,4.50,Honda,Jazz V
2,Chennai,2012,87000,Diesel,Manual,First,20.77,1248,6.00,Maruti,Ertiga VDI
3,Coimbatore,2013,40670,Diesel,Automatic,Second,15.20,1968,17.74,Audi,A4 New 2.0 TDI Multitronic
4,Jaipur,2013,86999,Diesel,Manual,First,23.08,1461,3.50,Nissan,Micra Diesel XV


In [6]:
dtf.describe()

,Year,Kilometers_Driven,Consommation,Engine,Price
count,5807.000000,5807.000000,5807.000000,5807.000000,5807.000000
mean,2013.335629,57986.480455,18.342332,1598.853453,8.615972
std,3.243129,38066.077924,4.060072,562.603091,8.605696
min,1998.000000,171.000000,7.810000,624.000000,0.440000
25%,2011.000000,34297.000000,15.500000,1198.000000,3.500000
50%,2014.000000,53392.000000,18.200000,1462.000000,5.550000
75%,2016.000000,73000.000000,21.100000,1968.000000,9.500000
max,2019.000000,775000.000000,28.400000,5461.000000,50.000000


In [23]:
dtf['Brand'].unique()

<StringArray>
[      'Hyundai',         'Honda',        'Maruti',          'Audi',
        'Nissan',        'Toyota',    'Volkswagen',          'Tata',
          'Land',    'Mitsubishi',       'Renault', 'Mercedes-Benz',
           'BMW',      'Mahindra',          'Ford',       'Porsche',
        'Datsun',        'Jaguar',         'Volvo',     'Chevrolet',
         'Skoda',          'Mini',          'Fiat',          'Jeep']
Length: 24, dtype: str

In [7]:
dtf.info()

<class 'pandas.DataFrame'>
RangeIndex: 5807 entries, 0 to 5806
Data columns (total 11 columns):
 #   Column             Non-Null Count  Dtype  
---  ------             --------------  -----  
 0   Location           5807 non-null   str    
 1   Year               5807 non-null   int64  
 2   Kilometers_Driven  5807 non-null   int64  
 3   Fuel_Type          5807 non-null   str    
 4   Transmission       5807 non-null   str    
 5   Owner_Type         5807 non-null   str    
 6   Consommation       5807 non-null   float64
 7   Engine             5807 non-null   int64  
 8   Price              5807 non-null   float64
 9   Brand              5807 non-null   str    
 10  Model              5807 non-null   str    
dtypes: float64(2), int64(3), str(6)
memory usage: 499.2 KB


In [8]:
model= jbl.load(r"model.pkl")

In [12]:
x,y = dtf.drop(columns=['Price']), dtf['Price']
X_train,X_test,y_train,y_test=train_test_split(x,y,test_size=0.2, random_state=42)

xdata = pd.DataFrame(X_test.join(y_test))

ypred=model.predict(X_test)
sc=r2_score(y_test, ypred)
rmse=np.sqrt(mean_squared_error(y_test, ypred))

print(f"R2 Score: {sc}")
print(f"RMSE: {rmse}")

xdata.head()

R2 Score: 0.9488432283520373
RMSE: 1.8803151524991968


,Location,Year,Kilometers_Driven,Fuel_Type,Transmission,Owner_Type,Consommation,Engine,Brand,Model,Price
501,Coimbatore,2017,49275,Diesel,Automatic,First,12.70,2179,Land,Rover Range Rover HSE Dynamic,45.64
3163,Mumbai,2016,9000,Petrol,Automatic,First,18.90,1197,Hyundai,Grand i10 Magna AT,5.45
3954,Coimbatore,2017,23249,Petrol,Automatic,First,19.16,2487,Toyota,Camry Hybrid 2.5,26.11
4275,Hyderabad,2012,86000,Diesel,Automatic,Second,14.70,1985,Volvo,XC60 D4 Summum,18.25
4898,Coimbatore,2017,21789,Diesel,Manual,First,24.30,1248,Maruti,Vitara Brezza ZDi,9.88


In [21]:
g=pd.DataFrame([{
    "Location": "Mumbai",
    "Year": 2016,
    "Kilometers_Driven": 9000,
    "Fuel_Type": "Petrol",
    "Transmission": "Automatic",
    "Owner_Type": "First",
    "Consommation": 18.9,
    "Engine": 1197,
    "Brand": "Hyundai",
    "Model": "Grand i10 Magna AT"
        
    }])

In [20]:
y=model.predict(g)
y

array([6.9783487], dtype=float32)

In [15]:
dtf.columns

Index(['Location', 'Year', 'Kilometers_Driven', 'Fuel_Type', 'Transmission',
       'Owner_Type', 'Consommation', 'Engine', 'Price', 'Brand', 'Model'],
      dtype='str')